# RAG retrieval check with turkish-rag-eval

Puts our students next to the baselines on a small Turkish RAG retrieval task: [RizgarOzan/turkish-rag-eval](https://github.com/RizgarOzan/turkish-rag-eval) (MIT; its data is CC BY-SA 4.0).
The bundled corpus is 54 Turkish Wikipedia health articles with 58 human-labelled questions (one annotator). Relevance is "the chunk comes from the answer's document and contains the answer span", so the same labels work for every chunking.

**Read the numbers with care.**
- 58 questions: the 95% intervals are about ±0.08 nDCG@10, so only large gaps are visible. This is a sanity check on a real retrieval task, not a leaderboard. The paired bootstrap below says which differences survive.
- The corpus is Wikipedia and so is much of our training mix; the contamination cell at the end measures the overlap.
- Retrieval only. The answer-quality part of the repo needs hosted LLM APIs and is left out for now.

Every model runs in its own process. Models without a Hugging Face folder of their own (our students) are read from Drive; baselines are downloaded at a pinned revision.

In [ ]:
import os, subprocess, sys, json, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_URL = 'https://github.com/RizgarOzan/turkish-rag-eval.git'
REPO_COMMIT = '900a8b067e14ade34c64548c9c43268548646463'   # main on 2026-10-28; the harness is under active development, so pin it
REPO_DIR = '/content/turkish-rag-eval'

os.environ['HF_HOME'] = '/content/hf_cache'   # local disk, never Drive
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as exc:
    print('HF_TOKEN not set (only needed for gated models):', exc)

if not os.path.isdir(REPO_DIR):
    !git clone --quiet {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} checkout --quiet {REPO_COMMIT}
!pip install --quiet -e '{REPO_DIR}[dense]'
!git -C {REPO_DIR} log -1 --oneline
print(os.path.exists(f'{REPO_DIR}/data/raw/corpus.json'), os.path.exists(f'{REPO_DIR}/data/eval/gold.json'))

## Models

`MODELS` maps a label to either a Hugging Face repo at a pinned revision (`hf` + `rev`) or a folder (`path`). `q` and `d` are the query and document prefixes: the harness has no prefix option of its own for most models (it knows E5 and EmbeddingGemma), so the runner registers them for each model. Models that expect an instruction get a second `_instr` entry; the plain entry is the protocol of the TR-MTEB authors.

Not included on purpose: voyage-4-nano (its custom code needs `trust_remote_code`, which the harness does not pass), Qwen3-Embedding-8B (needs a 24 GB GPU; add it with the prefix below if you have one).

In [ ]:
INSTR_8K = 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:'      # no space after "Query:" (model card)
LAB_INSTR = 'Instruct: Given a Turkish search text, retrieve relevant passages written in Turkish that best match the search text.\nQuery: '
RUN = lambda name: f'{DRIVE_ROOT}/runs/{name}/final'

MODELS = {
    # --- our students (all 1k base, 200K mix, plain protocol) ---
    'ours_pair_instr8k_s10800': dict(path=RUN('scale_two_instr8k_s10800_lr1e4/20261027-152516-b2b0c4b')),     # best: BGE + instructed 8K, 10,800 steps (TR-MTEB 67.71)
    'ours_pair_plain_s10800':   dict(path=RUN('scale_two_s10800_lr1e4/20261027-152514-b2b0c4b')),            # BGE + 8K, 10,800 steps (67.55)
    'ours_pair_plain_s5400':    dict(path=RUN('scale_two_lr1e4_mix200k_s5400/20261027-063854-79eb81a')),     # BGE + 8K, 5,400 steps (67.16)
    'ours_single_bge_s5400':    dict(path=RUN('scale_bge_mix200k_s5400_lr1e4/20261027-101819-77232d7')),     # BGE alone (66.07)
    'ours_single_8k_s5400':     dict(path=RUN('scale_8k_mix200k_s5400_lr1e4/20261027-105836-77232d7')),      # 8K-embed alone (65.67)
    # --- baselines ---
    'raw_base_1k':   dict(hf='ytu-ce-cosmos/modernbert-tr-base-1k', rev='c2e35f553101cce5b03fbb318ef8728473fd34dd'),                 # our student before distillation
    'lab_1k_embed':  dict(hf='ytu-ce-cosmos/modernbert-tr-1k-embed', rev='ba1e17bc7e95a257b4e614c300c4ca076711fce6'),
    'lab_1k_embed_instr': dict(hf='ytu-ce-cosmos/modernbert-tr-1k-embed', rev='ba1e17bc7e95a257b4e614c300c4ca076711fce6', q=LAB_INSTR),
    'lab_8k_embed':  dict(hf='ytu-ce-cosmos/modernbert-tr-embed', rev='855820e0079102805908d60d32cd9fd59c970b80'),
    'lab_8k_embed_instr': dict(hf='ytu-ce-cosmos/modernbert-tr-embed', rev='855820e0079102805908d60d32cd9fd59c970b80', q=INSTR_8K),
    'bge_m3':        dict(hf='BAAI/bge-m3', rev='5617a9f61b028005a4858fdac845db406aefb181'),
}
REFERENCE = 'ours_pair_instr8k_s10800'     # the paired comparisons below are against this model
TO_RUN = list(MODELS)                      # or a subset, e.g. ['ours_pair_instr8k_s10800', 'raw_base_1k']
for name, spec in MODELS.items():
    if 'path' in spec:
        assert os.path.isdir(spec['path']), f'{name}: {spec["path"]} not found'
print(len(MODELS), 'models')

## Run

Each model: materialise (download at the pinned revision, or use the Drive folder), run `turkish-rag-eval run` with the dense and hybrid (dense + BM25 with rank fusion) retrievers on all three chunkings, copy the results to Drive (`eval_rag/<label>/`). Finished models are skipped. Index embeddings are cached, so a re-run costs seconds.

In [ ]:
import time
from huggingface_hub import snapshot_download

RESULTS_LOCAL = '/content/rag_results'
RESULTS_DRIVE = f'{DRIVE_ROOT}/eval_rag'
CACHE_DIR = '/content/rag_embedding_cache'
os.makedirs(RESULTS_DRIVE, exist_ok=True)

RUNNER = r'''
import os
from turkish_rag_eval import models
path = os.environ["RAG_MODEL"]
models.MODEL_PREFIXES[path] = (os.environ["RAG_QPREFIX"], os.environ["RAG_DPREFIX"])
from turkish_rag_eval.cli import main
raise SystemExit(main(["run", "--model", path, "--retrievers", "dense", "hybrid_rrf",
                       "--out", os.environ["RAG_OUT"], "--embedding-cache", os.environ["RAG_CACHE"]]))
'''

def materialise(label, spec):
    if 'path' in spec:
        return spec['path']
    base = spec['hf'].replace('/', '__')
    local = f'/content/models/{base}_{spec["rev"][:8]}'
    if not os.path.isdir(local):
        snapshot_download(spec['hf'], revision=spec['rev'], local_dir=local)
    return local

failed = {}
for label in TO_RUN:
    spec = MODELS[label]
    done = Path(RESULTS_DRIVE) / label / 'summary.json'
    if done.exists():
        print(f'\n########## {label}: already done, skipped')
        continue
    print(f'\n########## {label}')
    t0 = time.time()
    try:
        model_path = materialise(label, spec)
        out = f'{RESULTS_LOCAL}/{label}'
        shutil.rmtree(out, ignore_errors=True)
        env = dict(os.environ, RAG_MODEL=model_path, RAG_QPREFIX=spec.get('q', ''), RAG_DPREFIX=spec.get('d', ''),
                   RAG_OUT=out, RAG_CACHE=CACHE_DIR)
        subprocess.run([sys.executable, '-c', RUNNER], cwd=REPO_DIR, env=env, check=True)
        shutil.copytree(out, f'{RESULTS_DRIVE}/{label}', dirs_exist_ok=True)
        print(f'=== {label} done in {(time.time() - t0) / 60:.1f} min')
    except Exception as exc:
        failed[label] = f'{type(exc).__name__}: {str(exc)[:300]}'
        print(f'=== {label} FAILED: {failed[label]}')
print('\nfailed:', failed or 'none')

## Results

nDCG@10 per model, retriever and chunking (the repo's headline metric), then Recall@5 and MRR, then the paired bootstrap: for every model the 95% interval of its nDCG@10 and of its difference to `REFERENCE`, over the labelled questions (10,000 resamples of the questions, the same questions for every model). A difference whose interval contains 0 is not resolved by 58 questions.

In [ ]:
import numpy as np, pandas as pd
sys.path.insert(0, f'{REPO_DIR}/src')
from turkish_rag_eval.metrics import ndcg_at_k

rows = []
for label in MODELS:
    path = Path(RESULTS_DRIVE) / label / 'summary.json'
    if path.exists():
        for r in json.load(open(path, encoding='utf-8')):
            rows.append({'model': label, 'retriever': r['retriever'], 'chunking': r['chunking'],
                         'ndcg@10': r['ndcg@10'], 'recall@5': r['recall@5'], 'mrr': r['mrr'], 'queries': r['queries_scored']})
df = pd.DataFrame(rows)
pd.set_option('display.width', 200)
for metric in ('ndcg@10', 'recall@5', 'mrr'):
    print(f'\n=== {metric}')
    print(df.pivot_table(index='model', columns=['retriever', 'chunking'], values=metric).round(3).reindex([m for m in MODELS if m in set(df.model)]))
print('\nqueries scored per model:', sorted(set(df.queries)))

In [ ]:
rng = np.random.default_rng(0)

def per_query_ndcg(label, chunking, retriever, k=10):
    data = json.load(open(Path(RESULTS_DRIVE) / label / f'perquery_{chunking}_{retriever}.json', encoding='utf-8'))
    scored = [q for q in data if q['total_relevant'] > 0]
    return [q['qid'] for q in scored], np.array([ndcg_at_k(q['relevance'], q['total_relevant'], k) for q in scored])

def interval(values, n=10000):
    idx = rng.integers(0, len(values), (n, len(values)))
    return np.percentile(values[idx].mean(axis=1), [2.5, 97.5])

for retriever in ('dense', 'hybrid_rrf'):
    for chunking in ('fixed', 'sentence', 'hierarchical'):
        if not (Path(RESULTS_DRIVE) / REFERENCE / f'perquery_{chunking}_{retriever}.json').exists():
            continue
        ref_ids, ref = per_query_ndcg(REFERENCE, chunking, retriever)
        print(f'\n=== {retriever} + {chunking}   (reference: {REFERENCE}, {len(ref)} questions)')
        print(f'{"model":<28}{"nDCG@10":>9}{"95% interval":>20}{"diff to ref":>13}{"95% interval":>20}')
        for label in MODELS:
            if not (Path(RESULTS_DRIVE) / label / f'perquery_{chunking}_{retriever}.json').exists():
                continue
            ids, x = per_query_ndcg(label, chunking, retriever)
            assert ids == ref_ids, f'{label}: different questions than the reference'
            lo, hi = interval(x)
            if label == REFERENCE:
                print(f'{label:<28}{x.mean():9.3f}{f"[{lo:.3f}, {hi:.3f}]":>20}')
            else:
                dl, dh = interval(x - ref)
                mark = '  *' if (dl > 0 or dh < 0) else ''
                print(f'{label:<28}{x.mean():9.3f}{f"[{lo:.3f}, {hi:.3f}]":>20}{x.mean() - ref.mean():+13.3f}{f"[{dl:+.3f}, {dh:+.3f}]":>20}{mark}')
print('\n* = the interval of the difference excludes 0')

## Training-data overlap

The repo warns that its Wikipedia corpus is probably in the training data of every model it ranks. For our students the text that can matter is the 200K mix (Wikipedia articles and Wikipedia QA passages are among its sources). This counts how many of the corpus's 13-grams (lower-cased word tokens) occur in the mix and which documents are affected. Overlap is exposure to the text; there are no question labels in our training data. It is a caveat to report, not proof of leakage.

In [ ]:
import re
from turkish_rag_eval.corpus import load_corpus

N = 13
docs = load_corpus(Path(REPO_DIR) / 'data/raw/corpus.json').docs
tok = lambda text: re.findall(r'\w+', text.lower())

owner, total = {}, {}
for d in docs:
    words = tok(d['text'])
    grams = {hash(tuple(words[i:i + N])) for i in range(len(words) - N + 1)}
    total[d['doc_id']] = len(grams)
    for g in grams:
        owner.setdefault(g, d['doc_id'])
print(f'{len(docs)} corpus documents, {len(owner)} distinct {N}-grams')

mix_path = f'{DRIVE_ROOT}/data/mix/mix_v2_200k.jsonl'
hit_texts, found = 0, {}
n_texts = 0
with open(mix_path, encoding='utf-8') as f:
    for line in f:
        n_texts += 1
        words = tok(json.loads(line)['text'])
        hits = {hash(tuple(words[i:i + N])) for i in range(len(words) - N + 1)} & owner.keys()
        if hits:
            hit_texts += 1
            for g in hits:
                found.setdefault(owner[g], set()).add(g)
covered = sum(len(v) for v in found.values())
print(f'{n_texts} mix texts; {hit_texts} share at least one {N}-gram with the RAG corpus')
print(f'{covered} of {len(owner)} corpus {N}-grams ({100 * covered / max(len(owner), 1):.2f}%) occur in the mix')
worst = sorted(((len(found.get(doc, ())) / max(t, 1), doc) for doc, t in total.items()), reverse=True)[:10]
print('most covered documents (share of their 13-grams found in the mix):')
for share, doc in worst:
    print(f'  {share:6.1%}  {doc}')

In [ ]:
from google.colab import runtime
runtime.unassign()